# Drive, github and requirements settings

In [84]:
from google.colab import drive
from google.colab import userdata


drive.mount('/content/drive')
KEY = userdata.get('KEY')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


set Github access token

In [2]:
import os
from google.colab import userdata
github_access_token = userdata.get('GITHUB_TOKEN')

# Replace 'your_token_here' with your actual token and 'your_repo_url_here' with your repository URL
os.environ['GITHUB_TOKEN'] = github_access_token

repo_url = 'https://github.com/sustaz/principle_of_law_detection.git'
modified_url = repo_url.replace('https://', f'https://{os.environ["GITHUB_TOKEN"]}@')

Clone Repository

In [3]:
!git clone {modified_url}

Cloning into 'principle_of_law_detection'...
remote: Enumerating objects: 69, done.
remote: Counting objects: 100% (69/69), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 69 (delta 35), reused 47 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (69/69), 186.18 KiB | 1.39 MiB/s, done.
Resolving deltas: 100% (35/35), done.


Set github credentials

In [ ]:
!chmod +x ./principle_of_law_detection/bash_commands/set_git_credentials.sh
!./principle_of_law_detection/bash_commands/set_git_credentials.sh

Syncronize notebook version

In [ ]:
!cp /content/drive/MyDrive/POLINE/gpt_notebook.ipynb /content/principle_of_law_detection/

In [ ]:
!cp /content/drive/MyDrive/POLINE/old_test_outputs.ipynb /content/principle_of_law_detection/

Add, commit, push code

In [ ]:
!chmod +x ./principle_of_law_detection/bash_commands/add_commit_push.sh
!./principle_of_law_detection/bash_commands/add_commit_push.sh "cleaned gpt notebook and updated library"

[main dbf5875] cleaned gpt notebook and updated library
 1 file changed, 1 insertion(+), 1 deletion(-)
 rewrite gpt_notebook.ipynb (96%)
Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 6.41 KiB | 1.28 MiB/s, done.
Total 3 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/sustaz/principle_of_law_detection.git
   5e86895..dbf5875  main -> main


Install requirements

In [4]:
!pip install -r '/content/principle_of_law_detection/requirements.txt'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.3/244.3 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.5/27.5 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 11.0 MB/s eta 0:00:00


# Import libraries

In [5]:
from principle_of_law_detection.src import gpt_utils as gu, utils as sr, text_preprocessing as tp, evaluation as ev
import json
import os
import pandas as pd

# Single experiments

## Single prompt approach

In [ ]:
def prompt_single_trial1(txt):
  return f"""
  Extract portions of the text from the argumentative part of the judgment that fit the definition of a JPOL.

    {txt}

    A JPOL (Judicial Principle of Law) should:

    Source and Content:
        Be a portion of text extracted from the argumentative part of a judgment.
        Contain the interpretation provided by the deciding court or an endorsed interpretation from a previous decision.

    Nature of Interpretation:
        Interpretate a rule, a general principle, or the consequences stemming from the application of a rule or principle within the legal system.
        Not be a rephrase of the legislation or a previous paragraph.
        Not concern the application of facts to the current case.
        Not be what the referring court asks.

    Citations and Endorsements:
        A JPOL can cite another JPOL.
        A JPOL can endorse a precedent of a European Court or an Advocate General's statement.

    Classify each paragraph with Y if the paragraph is a JPOL, N if the paragraph is not a JPOL.
    Avoid any explanation into the output.
    Use the following format for the output:
      Paragraph number: class

      Here are a few examples of what is NOT a JPOL:
  37
  On the contrary, the consequence of such a condition is to exclude altogether any reduction of the taxable amount for VAT purposes in the case of unpaid claims arising during the six-month period preceding the declaration of insolvency of the debtor company concerned, even where those claims become definitively irrecoverable at the end of the insolvency proceedings. Such automatic refusal of the right to a reduction is contrary to the principle of the neutrality of VAT.
  30
  As regards the context of which Article 132(1)(b) of the VAT Directive forms part, it is important to note that that provision must be read in the light of Article 134(a) of that directive, which requires, in any event, that the supply of goods or services concerned be essential to the transactions exempted within the scope of hospital and medical care (see, to that effect, judgments of 1 December 2005, Ygeia, C-394/04 and C-395/04, EU:C:2005:734, paragraph 26; of 14 June 2007, Horizon College, C-434/05, EU:C:2007:343, paragraph 38; and of 8 October 2020, Finanzamt D, C-657/19, EU:C:2020:811, paragraph 31).
  54
  In the light of the discretion enjoyed by the Member States in that context, as noted in paragraph 40 above, the Court has held that the existence of the option provided for in the first paragraph of Article 133 of the VAT Directive supports the interpretation that it is for the national law of each Member State to lay down the rules according to which such recognition may be granted to establishments which request it, even if the fact that a Member State has not exercised that option does not affect the possibility that an establishment may be recognised for the purposes of granting the exemption referred to in Article 132(1)(b) of the VAT Directive (see, to that effect, judgment of 6 November 2003, Dornier, C-45/01, EU:C:2003:595, paragraphs 64 to 66).

"""

system_prompt = "You are a judge with strong knowledge on tax law, expert about extracting Judicial Principles of Law (JPOLs) from legal judgments. You are very very skeptical and tend to say something is NOT a JPOL"

In [ ]:
def prompt_single_trial2(txt):
  return f"""
    1. A JPOL is a portion of text, extracted from the argumentative part of a judgement which contains the interpretation provided by the deciding court or provided in
     a previous decision and endorsed by the deciding court;
  This interpretation concerns a rule; or a general principle; or focuses on the consequences stemming from the application of a rule or a principle in a legal system.
  2. A JPOL can be a citation of another JPOL taken from a previous judgement AND
  3. A JPOL is not a rephrase of the legislation or a previous paragraph AND
  4. A JPOL is not a question concerning the application of facts to the current case AND
  5. A JPOL can be the endorsement of a precedent of a European Court AND
  6. A JPOL can be the endorsement of an Advocate General's statement AND
  7. A JPOL is not what the referring court asks.

  For each JPOL all the conditions must apply.

  Consider the following judgement, where each paragraph starts with a number:

  {txt}

  Check if each paragraph has the characteristics of a JPOL.

  Use the following format:
  Paragraph number: Y if JPOL.
  Paragraph number: N if not JPOL.
  Paragraph number: UND if it meets both criteria.

  Here are a few examples of what is NOT a JPOL:

  On the contrary, the consequence of such a condition is to exclude altogether any reduction of the taxable amount for VAT purposes in the case of unpaid claims arising during the six-month period preceding the declaration of insolvency of the debtor company concerned, even where those claims become definitively irrecoverable at the end of the insolvency proceedings. Such automatic refusal of the right to a reduction is contrary to the principle of the neutrality of VAT.

  As regards the context of which Article 132(1)(b) of the VAT Directive forms part, it is important to note that that provision must be read in the light of Article 134(a) of that directive, which requires, in any event, that the supply of goods or services concerned be essential to the transactions exempted within the scope of hospital and medical care (see, to that effect, judgments of 1 December 2005, Ygeia, C-394/04 and C-395/04, EU:C:2005:734, paragraph 26; of 14 June 2007, Horizon College, C-434/05, EU:C:2007:343, paragraph 38; and of 8 October 2020, Finanzamt D, C-657/19, EU:C:2020:811, paragraph 31).

  In the light of the discretion enjoyed by the Member States in that context, as noted in paragraph 40 above, the Court has held that the existence of the option provided for in the first paragraph of Article 133 of the VAT Directive supports the interpretation that it is for the national law of each Member State to lay down the rules according to which such recognition may be granted to establishments which request it, even if the fact that a Member State has not exercised that option does not affect the possibility that an establishment may be recognised for the purposes of granting the exemption referred to in Article 132(1)(b) of the VAT Directive (see, to that effect, judgment of 6 November 2003, Dornier, C-45/01, EU:C:2003:595, paragraphs 64 to 66).

  To recognise such an insurer as having that status would be tantamount to disregarding the principle of fiscal neutrality, since the VAT paid to the tax authorities would not be exactly proportional to the price actually received by the taxable customers who carried out the taxable transactions in question.
"""

system_prompt = ""

In [ ]:
def jpol_prompt_no_par_4_single(txt):
  prompt =  f"""

    Extract portions of the text from the argumentative part of the judgment that fit the definition of a JPOL.
    Only certain portions of the judgement text contain JPOLS, and these portions must be at least one complete sentence.
    A JPOL (Judicial Principle of Law) should fit all the following conditions:

    1. A JPOL is a portion of text, extracted from the argumentative part of a judgement which contains the interpretation provided by the deciding court or provided in
     a previous decision and endorsed by the deciding court; This interpretation concerns a rule; or a general principle; or focuses on the consequences stemming from the application of a rule or a principle in a legal system
    2. A JPOL can be a citation of another JPOL taken from a previous judgement AND
    3. A JPOL is not a rephrase of the legislation or a previous paragraph AND
    4. A JPOL is not a question concerning the application of facts to the current case AND
    5. A JPOL can be the endorsement of a precedent of a European Court AND
    6. A JPOL can be the endorsement of an Advocate General's statement AND
    7. A JPOL is not what the referring court asks.

    For each JPOL all the conditions must apply.

    #####

    Argumentative part of the judgment:

    {txt}

    #####
    Please extract a list of JPOLs using the format below.

    - A JPOL is a complete sentence found in the judgement.
    - It starts with a capital letter and ends with an end punctuation mark (.)

    #####

    Instructions for output:
    - Tag the portions of text that fit the JPOL definition with <JPOL> and </JPOL>.
    - Inside the <JPOL> tag, return only the first five and the last five words of the sentence.
    - Separate the first five and the last five words with ellipses (.....).

    Example output:
    If you have a sentence like this:
    This is just an example on how to tag a JPOL in a sentence.

    The tagged JPOL should be:
    <JPOL>This is just an example.....a JPOL in a sentence.</JPOL>

    """

  return prompt


system_prompt_no_par = "You are an expert about Judicial Principles of Law (JPOLs) from legal judgments."

In [ ]:
response = gu.ask_gpt_2(jpol_prompt_no_par_4_single(txt), system_prompt, KEY, model="gpt-4o", max_tokens=4000, temperature=0.2, top_p=1)

In [ ]:
txt = tp.extract_text_between_markers(txt)

In [ ]:
file = "/content/drive/MyDrive/POLINE/Dataset/Dataset_V1/Preprocessed_Judgements_Subset_noparagraph/Copia di VDP Dental Laboratory NV v Staatssecretaris van FinanciÃ«n a.xml"

with open(file) as f:
    file = f.read()

txt = "".join(file)

response = gu.ask_gpt_2(jpol_prompt_no_par_4_single(txt), system_prompt, KEY, model="gpt-4o", max_tokens=4000, temperature=0.2, top_p=1)

## LLM + Sim

In [ ]:
import pandas as pd

# Replace 'your_excel_file.xlsx' with the actual file path
jpol_labels = pd.read_excel('/content/drive/MyDrive/POLINE/JPOL_labels.xlsx')['LABELS'].to_list()

In [ ]:
def prompt_rag(txt):
  return f"""
          You are a professionist judge with strong knowledge on tax law.
          Your task is to identify JPOLS inside a text of a judgment.

          DEFINITION OF A JPOL:
              All JPOLs should Contain the interpretation provided by the deciding court or an endorsed interpretation from a previous decision.

              Nature of Interpretation:
                  - Interpretate a rule, a general principle, or the consequences stemming from the application of a rule or principle within the legal system.
                  - Not concern the application of facts to the current case.
                  - Not be what the referring court asks.

              Citations and Endorsements:
                  - A JPOL can cite another JPOL.
                  - A JPOL can endorse a precedent of a European Court or an Advocate General's statement.

          OUTPUT:
            The scope is to have as output a list of attributes that are useful, in a second phase, to find all the chunks in the text judgements
            that are JPOL, trough BM25 or cosine similarity. In the output include:
              - JPOL label
              - Motivation according to the definition
              - keywords

          TEXT OF THE JUDGEMENT:

          {txt}

            """


def prompt_rag_2(txt, jpol_labels):
  return f"""

          The scope is to have as output a list of attributes that are useful, in a second phase, to find all the chunks in the text judgements
          that are JPOL, trough BM25 or cosine similarity. In the output include:
            - JPOL label
            - Motivation according to the definition
            - keywords

            To assign the labels, use the following list of JPOLS LABELS:

            {jpol_labels}


          TEXT OF THE JUDGEMENT:

          {txt}

            """



system_prompt = """You are a professionist judge with strong knowledge on tax law.
                    Your task is to identify JPOLS inside a text of a judgment.

                    DEFINITION OF A JPOL:
                        All JPOLs should Contain the interpretation provided by the deciding court or an endorsed interpretation from a previous decision.

                        Nature of Interpretation:
                            - Interpretate a rule, a general principle, or the consequences stemming from the application of a rule or principle within the legal system.
                            - Not concern the application of facts to the current case.
                            - Not be what the referring court asks.

                        Citations and Endorsements:
                            - A JPOL can cite another JPOL.
                            - A JPOL can endorse a precedent of a European Court or an Advocate General's statement."""

In [ ]:
file_path = "/content/drive/MyDrive/POLINE/Dataset/Dataset_V1/Preprocessed_Judgements_Subset/Almos AgrÃ¡rkÃ¼lkereskedelmi Kft v Nemzeti AdÃ³- Ã©s VÃ¡mhiv.xml"

with open(file_path) as f:
    file = f.read()

txt = "".join(file)

response = gu.ask_gpt_2(prompt_rag_2(txt, jpol_labels), system_prompt, KEY, model="gpt-4o", max_tokens=4096, temperature=0.2, top_p=1)

In [ ]:
print(response)

### JPOL Identification and Analysis

#### JPOL 1
- **JPOL Label**: Concept of ‘subsidy directly linked to the price’
- **Motivation**: This interpretation clarifies the conditions under which a directive can be considered to have been correctly transposed into national law, emphasizing that the exact wording is not necessary as long as the directive's objectives are met in a clear and precise manner.
- **Keywords**: transposition, directive, national law, clear and precise manner, full application

#### JPOL 2
- **JPOL Label**: Concept of "non payment of the price"
- **Motivation**: This interpretation discusses the conditions under which Member States must reduce the taxable amount for VAT due to non-payment, cancellation, or reduction in price, highlighting the fundamental principle that VAT should only be charged on the amount actually received.
- **Keywords**: Article 90(1), VAT Directive, non-payment, cancellation, reduction in price, taxable amount

#### JPOL 3
- **JPOL Label**:

In [ ]:
import re


def extract_jpol_data(text):
    # Pattern to capture each JPOL block
    jpol_pattern = re.compile(r'### JPOL \d+\s*\n- \*\*JPOL Label\*\*: (.*?)\s*\n- \*\*Motivation\*\*: (.*?)\s*\n- \*\*Keywords\*\*: (.*?)(?=\n###|$)', re.DOTALL)

    # Find all matches
    matches = jpol_pattern.findall(text)

    # Create a list of dictionaries for each JPOL (Label, Motivation, Keywords)
    jpol_list = []
    for label, motivation, keywords in matches:
        jpol_dict = {
            "Label": label.strip(),
            "Motivation": motivation.strip(),
            "Keywords": [kw.strip() for kw in keywords.split(',')]
        }
        jpol_list.append(jpol_dict)

    return jpol_list

In [ ]:
extracted_jpols = extract_jpol_data(response)

In [ ]:
extracted_jpols

[{'Label': 'Concept of ‘subsidy directly linked to the price’',
  'Motivation': "This interpretation clarifies the conditions under which a directive can be considered to have been correctly transposed into national law, emphasizing that the exact wording is not necessary as long as the directive's objectives are met in a clear and precise manner.",
  'Keywords': ['transposition',
   'directive',
   'national law',
   'clear and precise manner',
   'full application']},
 {'Label': 'Concept of "non payment of the price"',
  'Motivation': 'This interpretation discusses the conditions under which Member States must reduce the taxable amount for VAT due to non-payment, cancellation, or reduction in price, highlighting the fundamental principle that VAT should only be charged on the amount actually received.',
  'Keywords': ['Article 90(1)',
   'VAT Directive',
   'non-payment',
   'cancellation',
   'reduction in price',
   'taxable amount']},
 {'Label': 'Debt which has become definitively

In [ ]:
import os
import xml.etree.ElementTree as ET
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
import re
import PyPDF2

def extract_text_from_pdf(pdf_path):
    # Open the PDF file
    with open(pdf_path, 'rb') as file:
        reader = PyPDF2.PdfReader(file)
        text = ""
        # Extract text from all pages
        for page_num in range(len(reader.pages)):
            text += reader.pages[page_num].extract_text()
    return text

def find_article(text, article_number, subsection=None):
    # Regular expression to match articles more robustly
    if subsection:
        pattern = rf'Article\s*{article_number}\s*\({subsection}\)[\s\S]+?(?=Article\s*\d+\s*\(?[a-z]?\)?\s|$)'
    else:
        pattern = rf'Article\s*{article_number}\s*[\s\S]+?(?=Article\s*\d+\s*\(?[a-z]?\)?\s|$)'

    match = re.search(pattern, text, re.IGNORECASE)
    if match:
        return match.group().strip()
    else:
        return f"Article {article_number} {'(' + subsection + ')' if subsection else ''} not found."



# Step 1: Parse XML and Extract Paragraphs
def parse_xml_to_paragraphs(file_path):
    with open(file_path, 'r') as file:
        # Using BeautifulSoup for parsing the XML file
        soup = BeautifulSoup(file, 'lxml')
        text_content = soup.get_text(separator=" ").strip()

        # Splitting the text into paragraphs based on the pattern (paragraph number)
        paragraphs = []
        current_paragraph = ""

        for line in text_content.splitlines():
            line = line.strip()
            if line.isdigit():  # It's a paragraph number
                if current_paragraph:
                    paragraphs.append(current_paragraph.strip())
                current_paragraph = ""
            else:
                current_paragraph += " " + line

        if current_paragraph:
            paragraphs.append(current_paragraph.strip())

    return paragraphs

# Step 2: Generate embeddings for paragraphs
def create_embeddings(paragraphs, model):
    return model.encode(paragraphs)

# Step 3: Create FAISS Index for Efficient Search
def create_faiss_index(embeddings):
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)
    return index

# Step 4: Search the most similar paragraph using the query
def search_similar_paragraph(query_embedding, index, paragraphs, k):
    D, I = index.search(query_embedding, k)  # Distance and index
    return [(paragraphs[idx], D[0][i]) for i, idx in enumerate(I[0])]

# Step 5: Create Query Embedding based on Label, Motivation, and Keywords
def create_query_embedding(label, motivation, keywords, model):
    query_text = f"{label}"
    return model.encode([query_text])


def cosine_similarity_search(query, tfidf_matrix, paragraphs, k):
    # Transform the query to match the same TF-IDF dimensions
    query_tfidf = vectorizer.transform([query])

    # Compute cosine similarity between the query TF-IDF vector and all paragraph TF-IDF vectors
    cosine_similarities = linear_kernel(query_tfidf, tfidf_matrix).flatten()


    # Get the indices of the paragraphs with the highest cosine similarities
    top_k_indices = cosine_similarities.argsort()[-k:][::-1]  # argsort sorts in ascending; reverse for descending

    # Return the top k similar paragraphs with their indices and similarities
    return [(paragraphs[idx], cosine_similarities[idx]) for idx in top_k_indices]


def get_max_score_texts(text_tuples):
    # Dictionary to hold unique texts with all their scores
    score_dict = {}

    # Iterate through each tuple in the list
    for text, score in text_tuples:
        if text in score_dict:
            # Append the score to the list of scores for this text
            score_dict[text].append(score)
        else:
            # Start a new list of scores for this text
            score_dict[text] = [score]

    # List to hold texts with their highest scores
    max_score_texts = []

    # Find the text with the maximum score among its duplicates
    for text, scores in score_dict.items():
        max_score = max(scores)  # Find the maximum score for the text
        max_score_texts.append((text, max_score))  # Append tuple of text and its max score

    return max_score_texts


def flatten_list(list_of_lists):
    # Using list comprehension to flatten the list
    return [element for sublist in list_of_lists for element in sublist]

In [ ]:
# Example usage
pdf_path = '/content/drive/MyDrive/POLINE/CELEX_32006L0112_EN_TXT.pdf'  # Adjust the path to your file
text = extract_text_from_pdf(pdf_path)
article_number = 138  # Replace with the desired article number
subsection = None  # Replace with the desired subsection, or None if you want the entire article
result = find_article(text, article_number, subsection)

print(result)

Article 138, goods dispatched or transported to a Member Stateother than that in which dispatch or transport of the goodsbegins are supplied VAT-exempt or where goods are transferredVAT-exempt to another Member State by a taxable person for thepurposes of his business, VAT shall become chargeable on the15th day of the month following that in which the chargeableevent occurs.
2. By way of derogation from paragraph 1, VAT shall become
chargeable on issue of the invoice provided for in Article 220, ifthat invoice is issued before the 15th day of the month followingthat in which the chargeable event occurs.
CHAPTER 3
Intra-Community acquisition of goods


In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

paragraphs = parse_xml_to_paragraphs(file_path)

# Create the TF-IDF model
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(paragraphs)

#paragraph_embeddings = create_embeddings(paragraphs, model)

#index = create_faiss_index(np.array(paragraph_embeddings))
similar_paragraphs = []

for jpol in extracted_jpols:

    label = jpol['Label']
    motivation = jpol['Motivation']
    keywords = jpol['Keywords']

    #query_embedding = create_query_embedding(label, motivation, keywords, model)

    #similar_paragraphs = search_similar_paragraph(query_embedding, index, paragraphs, k=5)

    query = f"{label}"
    similar_paragraphs.append(cosine_similarity_search(query, tfidf_matrix, paragraphs, 5))



flattened_results = flatten_list(similar_paragraphs)
results = get_max_score_texts(flattened_results)

# Output results
for text, score in results:
  print(f"Text: {text}\nScore: {score}\n")

Text: Secondly, it is important, on the other hand, that, for situations other than those linked to the non-payment of the price, national transposing provisions take into account all the situations in which, after a transaction has been concluded, part or all of the consideration has not been received by the taxable person, which is a matter for the national court to ascertain.
Score: 0.3189775567427321

Text: However, Article 90(2) permits Member States to derogate from the abovementioned rule in the case of total or partial non-payment of the transaction price. Hence taxable persons cannot rely, under Article 90(1) of the VAT Directive, on a right to a reduction of their taxable amount for VAT in the case of non-payment of the price if the Member State concerned intended to apply the derogation provided for in Article 90(2) of that directive.
Score: 0.5222436523443988

Text: It must be noted in that regard that, if the total or partial non-payment of the purchase price occurs withou

In [ ]:
flattened_results[1]

0.4263512099936439

#PREPROCESSING

In [38]:
from principle_of_law_detection.src import text_preprocessing as pr
import re


judgements_root = "/content/drive/MyDrive/POLINE/Dataset/Dataset_V1/Judgements_Subset" # file originali

judgements_root_output = "/content/drive/MyDrive/POLINE/Dataset/Dataset_V2/Judgements_Subset_by_paragraphs_expression"

os.makedirs(judgements_root_output, exist_ok=True)

judgements = os.listdir(judgements_root)

with open('/content/drive/MyDrive/POLINE/expressions.txt', 'r') as f:
  expressions = f.read().replace("\n", "").split("/")

for judgement in judgements:

  with open(os.path.join(judgements_root, judgement)) as f:
    file = f.read()

  txt = "".join(file)

  ## ESTRAI MOTIVAZIONE
  txt = pr.extract_text_between_markers(txt)


  ## STRUTTURA IN PARAGRAFI L'INPUT
  cleaned_paragraps = []
  # Find all the paragraphs
  #paragraphs = re.split(r'\n\n\d+\n\n', txt)

  # Define the regex pattern
  pattern = re.compile(r'(?<=\n\n)(\d+)')

  # Split the text using the regex pattern
  paragraphs = pattern.split(txt.strip())

  paragraphs_indexed = ['\n\nPARAGRAPH NUMBER : ' + paragraphs[i] + ' TEXT: ' + paragraphs[i+1] + '\n\n' for i in range(1, len(paragraphs), 2)]

  ## RIMUOVERE PARAGRAFI SICURAMENTE NON JPOL
  for par in paragraphs_indexed:

    par_flag = True

    for exp in expressions:
      if exp.lower() in par.lower():
        par_flag = False
        print(exp)
        break

    if par_flag:
      cleaned_paragraps.append(par.strip())
    else:
      continue

    # modulino che si può potenziare eventualmetne con similarity dividendo i chunk per phrases

  txt = "\n\n".join(cleaned_paragraps)

  with open(os.path.join(judgements_root_output, judgement), 'w') as f:
      f.write(txt)

the referring court asks
The referring court considers
the referring court asks
the referring court asks
the referring court asks
the referring court seeks to ascertain
the referring court seeks to ascertain
the referring court asks
the referring court asks
The referring court considers
the referring court asks


# MASSIVE EXPERIMENTS

In [33]:
def jpol_prompt(txt):
  prompt =  f"""Extract portions of the text from the argumentative part of the judgment that fit the definition of a JPOL.

    {txt}

    A JPOL (Judicial Principle of Law) should:

    Source and Content:
        Be a portion of text extracted from the argumentative part of a judgment.
        Contain the interpretation provided by the deciding court or an endorsed interpretation from a previous decision.

    Nature of Interpretation:
        Interpretate a rule, a general principle, or the consequences stemming from the application of a rule or principle within the legal system.
        Not be a rephrase of the legislation or a previous paragraph.
        Not be a question concerning the application of facts to the current case.
        Not be what the referring court asks.

    Citations and Endorsements:
    A JPOL can cite another JPOL.
    A JPOL can endorse a precedent of a European Court or an Advocate General's statement.

    Use the following format for the output:
      Paragraph number: Y if JPOL.
      Paragraph number: N if not JPOL."""

  return prompt


system_prompt = "You are an expert of judge with strong knowledge on jurisdiction and tax law. "

In [39]:
jsons_root = "/content/drive/MyDrive/POLINE/Annotazioni/poline_jsons/"
annotations_files = os.listdir(jsons_root)

In [40]:
judgements_root = "/content/drive/MyDrive/POLINE/Dataset/Dataset_V2/Judgements_Subset_by_paragraphs_expression"
judgemnts = os.listdir(judgements_root)

Experiments on whole text

In [108]:
prompt_name = "p_aspries_2"
responses = []

for idx, judgement in enumerate(judgemnts):

  with open(os.path.join(judgements_root, judgement)) as f:
    file = f.read()

  txt = "".join(file)

  response = gu.ask_gpt_2(jpol_prompt(txt), system_prompt, KEY, model="gpt-4o-2024-05-13", max_tokens=4000, temperature=0.2, top_p=1)

  result_root = f"/content/drive/MyDrive/POLINE/Results/preprocessed_input/Dataset_V2/full_responses_{prompt_name}_expressions_2"
  os.makedirs(result_root, exist_ok=True)
  sr.write_text_to_docx(response, os.path.join(result_root, f"{judgement}_full_response.docx"))

  responses.append((response, judgement))

Experiments per chunk  

In [ ]:
import re

prompt_name = "p_aspries_2"
responses = []
full_response_text = ""  # Initialize a variable to hold the full response

# Define how many paragraphs to process per chunk
chunk_size = 2

for idx, judgement in enumerate(judgemnts):

    with open(os.path.join(judgements_root, judgement)) as f:
        file = f.read()

    # Use a regular expression to split the text into paragraph number and text pairs
    paragraphs = re.findall(r'(PARAGRAPH NUMBER : \d+ TEXT: .+?)(?=PARAGRAPH NUMBER|\Z)', file, re.DOTALL)

    # Process the paragraphs in chunks
    for i in range(0, len(paragraphs), chunk_size):
        chunk = paragraphs[i:i + chunk_size]
        txt_chunk = "\n\n".join(chunk)  # Join paragraphs with paragraph number included for the current chunk

        response = gu.ask_gpt_2(jpol_prompt(txt_chunk), system_prompt, KEY, model="gpt-4o-2024-05-13", max_tokens=4000, temperature=0.2, top_p=1)

        # Save individual chunk response
        result_root = f"/content/drive/MyDrive/POLINE/Results/preprocessed_input/Dataset_V2/full_responses_{prompt_name}_chunk_2"
        os.makedirs(result_root, exist_ok=True)
        chunk_name = f"{judgement}_chunk_{i // chunk_size + 1}_full_response.docx"
        sr.write_text_to_docx(response, os.path.join(result_root, chunk_name))

        responses.append((response, judgement, chunk_name))

        # Append this chunk's response to the full response text
        full_response_text += response + "\n\n"  # Separate each chunk with a newline

    # Save the full combined response after all chunks have been processed
    full_response_name = f"{judgement}_full_combined_response.docx"
    sr.write_text_to_docx(full_response_text, os.path.join(result_root, full_response_name))

    responses.append((full_response_text, judgement, full_response_name))


In [43]:
def extract_answers(text):
    # Regular expression to find pairs of number and answer (Y or N)
    pattern = r'(\d+): (Y|N)'
    matches = re.findall(pattern, text)

    # Convert matches to list of tuples
    #pairs = [(int(num), answer) for num, answer in matches]

    return pd.DataFrame(matches, columns=['paragraph_number', 'label'])

Read responses from drive

In [102]:
import os
from docx import Document

def extract_texts_from_docx(folder_path):
    texts = []

    # Iterate through all files in the specified folder
    for filename in os.listdir(folder_path):
        if filename.endswith(".docx"):
            file_path = os.path.join(folder_path, filename)
            # Open the docx file and extract text
            doc = Document(file_path)
            file_text = "\n".join([para.text for para in doc.paragraphs])
            texts.append([file_text, filename.replace("_full_response.docx", "")])

    return texts

# Specify the path to your folder containing .docx files
folder_path = "/content/drive/MyDrive/POLINE/Results/preprocessed_input/Dataset_V2/full_responses_p_aspries_2_expressions_2"
responses = extract_texts_from_docx(folder_path)

Save results for full response

In [110]:
import re
dfs = []
for response, file_name in responses:
    dfs.append((extract_answers(response.replace("<", "").replace(">", "")), file_name[:5]))

results_df = sr.concatenate_dataframes(dfs)
results_df.to_excel(f"drive/MyDrive/POLINE/Results/preprocessed_input/Dataset_V2/full_responses_p_aspries_2_expressions_2/predictions/{prompt_name}.xlsx", index=False)

Save results for chunked response

In [ ]:
import re
dfs = []
for response, file_name, _ in responses:
    dfs.append((extract_answers(response.replace("<", "").replace(">", "")), file_name[:5]))

results_df = sr.concatenate_dataframes(dfs)
results_df.to_excel(f"drive/MyDrive/POLINE/Results/preprocessed_input/Dataset_V2/full_responses_p_aspries_2_chunk_2/predictions/{prompt_name}_.xlsx", index=False)

In [111]:
results_df

,paragraph_number,label,file_name
0,19,N,Minis
1,20,N,Minis
2,21,N,Minis
3,22,N,Minis
4,23,N,Minis
...,...,...,...
289,67,N,Micha
290,68,Y,Micha
291,69,N,Micha
292,70,N,Micha


# EVALUATION STRUCTURED IN PARAGRAPHS

In [50]:
jsons_root = "/content/drive/MyDrive/POLINE/Annotazioni/poline_jsons/"
annotations_files = os.listdir(jsons_root)

In [51]:
judgements_root = "/content/drive/MyDrive/POLINE/Dataset/Dataset_V2/Judgements_Subset_by_paragraphs_expression"
judgemnts = os.listdir(judgements_root)

Read GT from Json

In [52]:
ground_truths = []

for ann_file in annotations_files:

  ann_dict = json.load(open(os.path.join(jsons_root,ann_file)))


  for ann in ann_dict['annotations']:
    paragraph_number = ann['text'].split()[0]
    # split_par = str(int(paragraph_number) + 1)
    # txt_par = txt.split(split_par)
    label = ann['type']
    file_name = ann_file[:5]

    ground_truths.append((file_name, paragraph_number, label))

gt_batch1 = pd.DataFrame(ground_truths, columns=['file_name', 'paragraph_number', 'ground_truth'])

Read GT from excel

In [53]:
gt_batch2 = pd.read_excel('/content/drive/MyDrive/POLINE/Annotazioni/CJUE_Taxable_Amount_Addendum_Piera.xlsx')
gt_batch2['file_name'] = gt_batch2['file_name'].apply(lambda x: x[:5])
gt_batch2['ground_truth'] = 'JPOL'

In [54]:
ground_truth_df = pd.concat([gt_batch1, gt_batch2]).drop('other', axis=1)

Merge annotations with original files to have all NOT JPOLS paragraphs

In [55]:
import os
import pandas as pd

# Folder containing text files
folder_path = 'drive/MyDrive/POLINE/Dataset/Dataset_V2/Judgements_Subset_by_paragraphs'

def merge_annotations_with_files(folder_path, df):
  # Function to parse text files
  def parse_paragraphs(file_path):
      paragraphs = {}
      with open(file_path, 'r') as f:
          content = f.read().split('PARAGRAPH NUMBER : ')
          for part in content[1:]:
              number, text = part.split('TEXT:', 1)
              paragraphs[int(number.strip())] = text.strip()
      return paragraphs

  # List to collect new rows
  new_rows = []

  # Iterate over each file
  for file_name in df['file_name'].unique():
      # Get the corresponding text file
      matching_files = [f for f in os.listdir(folder_path) if f.startswith(file_name[:5])]

      for txt_file in matching_files:
          txt_file_path = os.path.join(folder_path, txt_file)

          # Parse the paragraphs from the text file
          paragraphs = parse_paragraphs(txt_file_path)

          # Get the paragraph numbers already in the DataFrame for this file
          existing_paragraphs = df[df['file_name'] == file_name]['paragraph_number'].tolist()

          # Find missing paragraphs
          actual_paragraphs = {str(num) for num in paragraphs.keys()}
          missing_paragraphs = set(actual_paragraphs) - set(existing_paragraphs)

          print(missing_paragraphs)
          # Add missing paragraphs to the list of new rows
          for paragraph_number in missing_paragraphs:
              new_row = {
                  'file_name': file_name,
                  'paragraph_number': paragraph_number,
                  'ground_truth': 'not_JPOL'
              }
              new_rows.append(new_row)

  # Create a DataFrame from the new rows and concatenate it to the original DataFrame
  if new_rows:
      df_new = pd.DataFrame(new_rows)
      gt_df = pd.concat([df, df_new], ignore_index=True)

  return gt_df

# Sort the dataframe by file_name and paragraph_number
ground_truth_df = merge_annotations_with_files(folder_path, gt_batch1)

{'37', '14', '23', '22', '43', '15', '33', '17', '34', '7', '45', '2', '40', '42', '39', '44', '24', '12', '8', '18', '20', '1', '6', '3', '9', '38', '13', '30', '10', '19', '11', '21', '36', '48', '26', '46', '35', '4', '5', '16'}
{'37', '60', '14', '23', '22', '41', '65', '62', '15', '33', '17', '7', '45', '56', '2', '40', '42', '57', '25', '24', '12', '8', '51', '18', '20', '6', '1', '32', '3', '9', '28', '49', '13', '30', '10', '19', '21', '11', '48', '29', '63', '26', '27', '46', '53', '35', '4', '5', '16'}
{'37', '22', '62', '64', '54', '40', '25', '57', '52', '12', '8', '51', '20', '6', '38', '9', '67', '30', '10', '21', '29', '27', '35', '5', '16', '14', '23', '43', '15', '33', '17', '34', '7', '31', '56', '2', '24', '18', '1', '32', '3', '28', '58', '47', '13', '36', '19', '11', '48', '50', '26', '46', '4'}
{'14', '23', '22', '15', '17', '7', '31', '2', '39', '44', '25', '24', '12', '8', '18', '20', '1', '6', '3', '38', '9', '13', '10', '19', '11', '21', '27', '46', '4', '5', 

In [142]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

def compute_metrics(ground_truth_df, results_df):
    res_df = results_df.copy()
    gt_df = ground_truth_df.copy()

    gt_df['paragraph_number'] = gt_df['paragraph_number'].astype(str)
    res_df['paragraph_number'] = res_df['paragraph_number'].astype(str)

    # Map 'JPOL' to 'JPOL' and 'OTHER' to 'not_JPOL'
    res_df['label'] = res_df['label'].map({'Y': 'JPOL', 'N': 'not_JPOL'})
    #res_df['label'] = res_df['label'].map({'JPOL': 'JPOL', 'OTHER': 'not_JPOL'})

    # Merge the two DataFrames on 'paragraph_number' and 'file_name'
    merged_df = pd.merge(res_df, gt_df, on=['paragraph_number', 'file_name'], how='outer')

    # Fill missing predictions (in the 'label' column) with 'not_JPOL'
    #merged_df['label'] = merged_df['label'].fillna('not_JPOL')

    # Create binary labels for metric calculations
    merged_df['ground_truth_binary'] = merged_df['ground_truth'].apply(lambda x: 1 if x == 'JPOL' else 0)
    merged_df['predicted_binary'] = merged_df['label'].apply(lambda x: 1 if x == 'JPOL' else 0)

    # Function to calculate the metrics for each group (by file_name)
    def calculate_metrics(group):
        precision = precision_score(group['ground_truth_binary'], group['predicted_binary'], zero_division=0)
        recall = recall_score(group['ground_truth_binary'], group['predicted_binary'], zero_division=0)
        f1 = f1_score(group['ground_truth_binary'], group['predicted_binary'], zero_division=0)
        accuracy = accuracy_score(group['ground_truth_binary'], group['predicted_binary'])
        return pd.Series({'precision': precision, 'recall': recall, 'f1': f1, 'accuracy': accuracy})

    # Calculate metrics for each file_name group
    metrics_by_filename = merged_df.groupby('file_name').apply(calculate_metrics).reset_index()

    return metrics_by_filename, merged_df


In [143]:
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

def compute_total_metrics(ground_truth_df, results_df):
    res_df = results_df.copy()
    gt_df = ground_truth_df.copy()

    gt_df['paragraph_number'] = gt_df['paragraph_number'].astype(str)
    res_df['paragraph_number'] = res_df['paragraph_number'].astype(str)

    # Map 'JPOL' to 'JPOL' and 'OTHER' to 'not_JPOL'
    #res_df['label'] = res_df['label'].map({'JPOL': 'JPOL', 'OTHER': 'not_JPOL'})
    res_df['label'] = res_df['label'].map({'Y': 'JPOL', 'N': 'not_JPOL'})

    # Merge the two DataFrames on 'paragraph_number' and 'file_name'
    merged_df = pd.merge(res_df, gt_df, on=['paragraph_number', 'file_name'], how='outer')

    # Fill missing predictions (in the 'label' column) with 'not_JPOL'
    #merged_df['label'] = merged_df['label'].fillna('not_JPOL')

    # Create binary labels for metric calculations
    merged_df['ground_truth_binary'] = merged_df['ground_truth'].apply(lambda x: 1 if x == 'JPOL' else 0)
    merged_df['predicted_binary'] = merged_df['label'].apply(lambda x: 1 if x == 'JPOL' else 0)

    # Calculate precision, recall, and f1-score
    precision = precision_score(merged_df['ground_truth_binary'], merged_df['predicted_binary'], zero_division=0)
    recall = recall_score(merged_df['ground_truth_binary'], merged_df['predicted_binary'], zero_division=0)
    f1 = f1_score(merged_df['ground_truth_binary'], merged_df['predicted_binary'], zero_division=0)
    accuracy = accuracy_score(merged_df['ground_truth_binary'], merged_df['predicted_binary'])

    return precision, recall, f1, accuracy

In [144]:
results_df = pd.read_excel("drive/MyDrive/POLINE/Results/preprocessed_input/Dataset_V1/predictions/p_aspries_2_predictions.xlsx")
#results_df = pd.read_excel("drive/MyDrive/POLINE/Results/preprocessed_input/Dataset_V2/full_responses_p_aspries_2_chunk_2/predictions/p_aspries_2_.xlsx")
#results_df = pd.read_excel("/content/drive/MyDrive/POLINE/Results/preprocessed_input/Dataset_V2/full_responses_p_aspries_2_expressions/predictions/p_aspries_2.xlsx")
#results_df = pd.read_excel("/content/drive/MyDrive/POLINE/Results/preprocessed_input/Dataset_V2/full_responses_p_aspries_2_expressions_2/predictions/p_aspries_2.xlsx")

# Chiamare la funzione
metrics_by_filename, merged_df = compute_metrics(ground_truth_df, results_df)

# Mostrare i risultati
print(metrics_by_filename)

precision, recall, f1, accuracy = compute_total_metrics(ground_truth_df, results_df)

print(f'<--------------------->')

print(f'Total F1-Score: {f1:.2f}')
print(f'Total Accuracy: {accuracy:.2f}')

   file_name  precision    recall        f1  accuracy
0      A & G   1.000000  1.000000  1.000000  1.000000
1      Autor   0.700000  0.777778  0.736842  0.883721
2      Boehr   0.608696  1.000000  0.756757  0.865672
3      CS an   0.800000  0.923077  0.857143  0.931034
4      DNB B   0.928571  0.812500  0.866667  0.914894
5      ELVOS   0.533333  1.000000  0.695652  0.854167
6      Euler   0.636364  1.000000  0.777778  0.911111
7      Finan   0.600000  0.937500  0.731707  0.830769
8      I Gmb   0.736842  0.666667  0.700000  0.857143
9      Micha   1.000000  0.687500  0.814815  0.931507
10     Minis   0.823529  1.000000  0.903226  0.930233
<--------------------->
Total F1-Score: 0.80
Total Accuracy: 0.89


<ipython-input-142-20db9e1c1a27>:33: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  metrics_by_filename = merged_df.groupby('file_name').apply(calculate_metrics).reset_index()


In [ ]:
metrics_by_filename['total precision'] = precision
metrics_by_filename['total recall'] = recall
metrics_by_filename['total f1'] = f1

In [ ]:
metrics_by_filename.to_excel("/content/drive/MyDrive/POLINE/Results/preprocessed_input/metrics_evaluation/Piera_e_Alessia 24.06.24_1_predictions.xlsx")

# EVALUATION UNSTRUCTURED

TODO BILLI:

*   Destruttura sentenze

TODO ASPRO:

*   Metriche AreaJPOL/AreaTesto
*   Map "5 words tags to paragraph" per comparare approccio strutturato e non






JPOL Ratio= Total number of words in the text / Total number of words tagged as JPOL​

\text{JPOL Ratio} = \frac{W_{\text{JPOL}}}{W_{\text{total}}}


Extract JPOL Sections: Use regex to find the text within JPOL tags.
Count JPOL Words: Split the JPOL text into parts around the dots. Count the words in the first and last parts.
Count Total Words: Remove JPOL tags from the total text and count the remaining words.
Add JPOL Word Count: Add the JPOL word count to the total word count.
Calculate Ratio: Compute the ratio of JPOL words to the total words.

In [ ]:
import re

def calculate_jpol_ratio(tagged_phrase, total_text, jpol_tag='<JPOL>', jpol_end_tag='</JPOL>'):
    # Extract JPOL sections
    jpol_texts = re.findall(f'{jpol_tag}(.*?){jpol_end_tag}', tagged_phrase, re.DOTALL)

    # Split JPOL text into beginning and end parts
    jpol_word_count = 0
    for jpol in jpol_texts:
        # Split into parts around the dots
        parts = jpol.split('...')
        if len(parts) == 2:
            # Count words in the first and last parts
            first_part_words = len(parts[0].split())
            last_part_words = len(parts[1].split())
            jpol_word_count += first_part_words + last_part_words

    # Remove JPOL tags from text to count total words
    clean_text = re.sub(f'{jpol_tag}.*?{jpol_end_tag}', '', total_text, flags=re.DOTALL)
    total_word_count = len(clean_text.split())

    # Add the JPOL word count to the total word count
    total_word_count += jpol_word_count

    # Calculate JPOL Ratio
    if total_word_count == 0:
        return 0

    jpol_ratio = jpol_word_count / total_word_count
    return round(jpol_ratio, 2)

In [ ]:
from docx import Document

f = open("/content/drive/MyDrive/POLINE/Results/preprocessed_input/full_responses_billaspries_4_no_par/A & G Fahrschul-Akademie GmbH v Finanzamt Wolfenbüttel.xml_full_response.docx", 'rb')
document = Document(f)
f.close()

jpols_text = document.paragraphs[0].text

file = "/content/drive/MyDrive/POLINE/Dataset/Dataset_V1/Judgements_Subset/A & G Fahrschul-Akademie GmbH v Finanzamt Wolfenbüttel.xml"

with open(file) as f:
    file = f.read()

judg_text = "".join(file)

judg_text = tp.extract_text_between_markers(judg_text)

In [ ]:
judg_text

'The first question\n\n16\n\nBy its first question, the referring court asks, in essence, whether the concept of ‘school or university education’, within the meaning of Article 132(1)(i) and (j) of Directive 2006/112, must be interpreted as covering motor vehicle driving tuition provided by a driving school, such as that at issue in the main proceedings, for the purpose of acquiring driving licences for vehicles in categories B and C1 referred to in Article 4(4) of Directive 2006/126.\n\n17\n\nArticle 132 of Directive 2006/112 provides for exemptions which, as indicated by the title of the chapter in which that provision features, are intended to encourage certain activities in the public interest. However, those exemptions do not cover every activity performed in the public interest, but only those listed in that provision and described in great detail (judgment of 4 May 2017, Brockenhurst College, C-699/15, EU:C:2017:344, paragraph 22 and the case-law cited).\n\n18\n\nAccording to th

In [ ]:
ratio = calculate_jpol_ratio(jpols_text, judg_text)
print(ratio)

0.13


# Paragraph Division Testing

In [ ]:
import pandas as pd

# Replace 'your_excel_file.xlsx' with the actual file path
jpol_labels = pd.read_excel('/content/drive/MyDrive/POLINE/JPOL_labels.xlsx')['LABELS'].to_list()

In [ ]:
file = "/content/drive/MyDrive/POLINE/Dataset/Dataset_V1/Preprocessed_Judgements_Subset_noparagraph/Finanzamt B v X-Beteiligungsgesellschaft mbH.xml"

with open(file) as f:
    file = f.read()

txt = "".join(file)


In [ ]:
file = "/content/drive/MyDrive/POLINE/Dataset/Dataset_V1/Judgements_Ita_Subset/Cass. 2020_25523.rtf"

with open(file) as f:
    file = f.read()

txt2 = "".join(file)

In [ ]:
def prompt_par_div(txt, jpol_labels):
  return f"""

          The scope is to have as output a list of paragraphs that are useful, in a second phase, to find all the chunks in the text judgements.
          Divide the judgements into paragraphs, following the reasoning of the judge, and these criteria:

          Sentence Boundaries: Each paragraph should consist of complete sentences. Use periods (.), question marks (?), and exclamation marks (!) to identify the end of a sentence.
          Clause Identification: Break paragraphs at complex or compound sentences with multiple clauses, especially when separated by conjunctions like "however," "therefore," or "furthermore." Each clause that introduces a new point can often justify starting a new paragraph.
          Quotation Marks: Whenever there are quoted texts, especially legal references or external opinions (e.g., quotations from laws, previous judgments, or expert opinions), they should generally start a new paragraph.
          Lists and Enumerations: Numbered or bullet-pointed lists (like Article or Clause numbers in legal texts) should be isolated into their own paragraphs.
          Structural Headings and Subsections: Break paragraphs when a new section, heading, or subheading begins (e.g., "Legal context," "Consideration of the questions referred"). These serve as natural divisions.
          Punctuation Markers: Look for colons (:) and semicolons (;) within sentences. Text following a colon often starts a new explanatory idea and can signal a paragraph break.
          Legal References: Any reference to a law, case, or directive (e.g., "Article 132(1)(f) of Directive 2006/112/EC") should be placed at the start of a new paragraph if it introduces a new point or argument.

          The entire text must be divided.

          TEXT OF THE JUDGEMENT:

          {txt}

          Use the following format:
          Paragraph number: #
          Text: text

          """

system_prompt = "You are an expert of judge with strong knowledge on jurisdiction and tax law. "

In [ ]:
response = gu.ask_gpt_2(prompt_par_div(txt, jpol_labels), system_prompt, KEY, model="gpt-4o", max_tokens=4000, temperature=0.2, top_p=1)

In [ ]:
print(response)

Certainly! Here is the text of the judgment divided into paragraphs according to the specified criteria:

---

**Paragraph number: 1**  
**Text:** Admissibility of the questions referred

---

**Paragraph number: 2**  
**Text:** As is clear from the facts, as set out by the referring court, the case in the main proceedings concerns a supply of services executed in full by the end of 2012 and for which the consideration was paid, pursuant to the contract, in annual instalments during the following five years.

---

**Paragraph number: 3**  
**Text:** In its observations before the Court, the defendant in the main proceedings disputes that statement of the facts. In support of its objection, it relies on a document which supports provisional findings made by a German court in a case other than that in the main proceedings.

---

**Paragraph number: 4**  
**Text:** However, such a document cannot call into question the statement of the facts as set out in the order for reference. It is su

In [ ]:
txt3 = """
Motivi della decisione
che:
- con il primo motivo il ricorrente denuncia in relazione all'art. 360  c.p.c., comma 1, n. 3, la violazione e falsa applicazione della L. n. 212 del 2000 , art. 7 , del D.P.R. n. 600 del 1973 , art. 42 , e del D.P.R. n. 633 del 1972 , art. 56 , per avere la CTR erroneamente rigettato il motivo di appello relativo alla dedotta carenza motivazionale dell'avviso di accertamento in questione per mancata allegazione ad esso delle valutazioni OMI e dei listini camerali (trattandosi di "documenti di pubblico dominio, riportati in pubblicazioni periodiche consultabili da qualunque interessato"), ancorchè, ad avviso del ricorrente, la semplice conoscibilità degli atti richiamati nell'atto impositivo non fosse sufficiente a integrare gli estremi di una corretta motivazione, dovendo le determinazioni dell'Amministrazione essere portate pienamente a conoscenza del contribuente al momento della notifica dell'atto impositivo e non essendo ammessa alcuna successiva integrazione di conoscenza, tanto più che i dati cui l'avviso, nella specie, aveva rinviato (valutazioni OMI e tabelle tenute dalle Camere di commercio) non erano, al tempo, neppure accessibili (essendo alla data de 23/7/2009 di notifica dell'avviso di accertamento, relativo all'anno 2005, disponibili sul sito OMI solo i valori del secondo semestre 2007 - primo semestre 2009 mentre gli altri dati erano disponibili dietro oneroso pagamento);
- il motivo è infondato;
- va innanzitutto premesso che è pacifico che la motivazione dell'avviso di accertamento contenga il riferimento ad atti esterni, quali "le valutazioni OMI e i listini camerali", ad esso non allegati. Dispone la L. n. 212 del 2000 , art. 7 , comma 1, ultimo periodo, a proposito degli atti dell'Amministrazione finanziaria, che: "Se nella motivazione si fa riferimento ad un altro atto, questo deve essere allegato all'atto che lo richiama.". In materia di accertamenti in rettifica ed accertamenti d'ufficio, prevede, a sua volta, il D.P.R. n. 600 del 1973 , art. 42 , comma 2, ultimo periodo, come modificato dal D.Lgs. 26 gennaio 2001, n. 32 , art. 1 , comma 1, lett. c), nella versione vigente ratione temporis, che: " Se la motivazione fa riferimento ad un altro atto non conosciuto nè ricevuto dal contribuente, questo deve essere allegato all'atto che lo richiama salvo che quest'ultimo non ne riproduca il contenuto essenziale.". Il successivo comma dispone infine che: "L'accertamento è nullo se l'avviso non reca la sottoscrizione, le indicazioni, la motivazione di cui al presente articolo e ad esso non è allegata la documentazione di cui al comma 2, ultimo periodo.";
- come da ultimo ricordato nella sentenza n. 1252 del 2020, nella giurisprudenza di questa Corte, molteplici sono le pronunce che, al fine di soddisfare il requisito della motivazione dell'accertamento, hanno ritenuto sufficiente che l'atto esterno, richiamato da quello impositivo, fosse, se non effettivamente conosciuto, quanto meno conoscibile dal contribuente destinatario dell'avviso. Non si intende, in questo senso, far riferimento alle affermazioni giurisprudenziali relative alla conoscibilità di atti richiamati, già oggetto di precedente notificazione al contribuente (Cass. 25/07/2012, n. 13110 ), o sottoposti a pubblicità legale (Cass. 19/12/2014, n. 27055 , in motivazione), trattandosi di ipotesi accomunabili dall'operatività di presunzioni legali (per quanto diversificate) di conoscenza, e quindi di equiparazione ex lege della conoscibilità alla conoscenza;
- piuttosto, ci si riferisce a quelle pronunce che hanno ritenuto legittima anche la motivazione per relationem che richiami, senza allegarli, atti che si possano presumere, solo iuris tantum, conosciuti dal destinatario dell'accertamento (Cass. 17/12/2014, n. 26527 ; Cass. 27/11/2015, n. 24254 ; Cass. 30/10/2018,n. 27628 ). E, soprattutto, ci si richiama a quell'orientamento che, finanche nel caso di doppia motivazione per relationem, ovvero quando il documento menzionato nella motivazione dell'atto tributario faccia a sua volta riferimento ad ulteriori documenti, ritiene sufficiente che questi ultimi siano, se non in possesso o comunque conosciuti dal contribuente, quanto meno agevolmente conoscibili da quest'ultimo (Cass. 12/12/2018, n. 32127 ; Cass. 24/11/2017, n. 28060 ; Cass. 04/06/2018, n. 14275 , ex plurimis, in tema di avviso di accertamento dei redditi del socio che rinvii a quello riguardante i redditi della società, ancorchè solo a quest'ultima notificato; Cass. 17/05/2017, n. 12312 , ex plurimis, relativa all'accertamento del maggior valore dell'immobile sulla base dei prezzi medi evincibili dal listino della Borsa immobiliare dell'Umbria, pubblicato dalla locale camera di Commercio ed agevolmente reperibile dalla contribuente);
- infatti, deve ritenersi che l'interpretazione giurisprudenziale della L. n. 212 del 2000 , art. 7 , comma 1, ultimo periodo, e del D.P.R. n. 600 del 1973 , art. 42 , comma 2, ultimo periodo e del D.P.R. n. 600 del 1973 , art. 42 , comma 3, nel senso che non sia nullo l'accertamento la cui motivazione fa riferimento ad un altro atto ad esso non allegato, ma conoscibile agevolmente dal contribuente, realizzi un adeguato bilanciamento tra le esigenze di economia dell'azione amministrativa (e quindi di buon andamento dell'amministrazione, ex art. 97  Cost.) - che giustificano l'ammissibilità, anche normativa, della motivazione per relationem (sul punto cfr. Cass. 29/01/2008, n. 1906 , in motivazione) - ed il pieno esercizio del diritto di difesa del contribuente (rilevante ex artt. 24  e 111  Cost.) nel giudizio di impugnazione dell'atto impositivo, che sarebbe illegittimamente compresso se la conoscibilità dell'atto esterno richiamato dalla motivazione non fosse agevole, ma richiedesse un'attività di ricerca complessa (Cass. n. 1252 del 2020 , cit.);
- nella specie, la CTR ha fatto buon governo dei suddetti principi nel ritenere che la mancata allegazione all'avviso di accertamento in questione dei prezziari OMI e dei listini camerati non implicasse una carenza motivazionale dell'atto impositivo medesimo in quanto trattavasi di documenti di "pubblico dominio", riportati in pubblicazioni periodiche consultabili da qualunque interessato; con ciò significando che tali dati cui rinviava l'avviso de quo rientrassero nella sfera quantomeno di agevole conoscibilità da parte del contribuente con conseguente sufficienza motivazionale dello stesso;
- con il secondo motivo, il ricorrente denuncia, in relazione all'art. 360  c.p.c., comma 1, n. 3, la violazione e falsa applicazione del D.P.R. n. 600 del 1973 , art. 39 , comma 1, lett. d), e dell'art. 2729  c.c., per avere la CTR ritenuto erroneamente sussistenti nella fattispecie elementi indiziari della sottofatturazione - dotati dei caratteri di gravità, precisione e concordanza - ancorchè, oltre alle valutazioni OMI e alle tabelle camerali, l'Agenzia delle entrate avesse fatto riferimento nella ricostruzione dell'effettivo corrispettivo di vendita a un atto preliminare privo di sottoscrizione di entrambi i contraenti, e pertanto inesistente, peraltro riferito alla vendita di una diversa costruzione immobiliare;
- il motivo si profila in parte inammissibile e in parte infondato;
- va premesso che l'accertamento fiscale da cui muove la presente controversia, è un accertamento di tipo analitico - induttivo, con cui il fisco procede alla rettifica di singoli componenti reddituali, ancorchè di rilevante importo, consentito, ai sensi del D.P.R. 29 settembre 1973, n. 600 , art. 39 , comma 1, lett. d), pure in presenza di contabilità formalmente tenuta, giacchè la disposizione presuppone, appunto, scritture regolarmente tenute e, tuttavia, contestabili in forza di valutazioni condotte sulla base di presunzioni gravi, precise e concordanti che facciano seriamente dubitare della completezza e fedeltà della contabilità esaminata (ex multis: Cass., sez. 5, n. 33508 del 2018 ; n. 20060 del 2014 );
- in particolare, ai sensi del D.P.R. n. 131 de 1986, art. 52, commi 4 e 5, a decorrere dal 10 luglio 1986, il potere di rettifica dei valori dichiarati negli atti era impedito qualora gli stessi fossero risultati pari o superiori a quel minimum determinato dalla capitalizzazione delle rendite catastali - che si otteneva moltiplicando per specifici coefficienti fissi di legge il valore catastale - con l'unico limite dato dall'eventuale individuazione, da parte dell'Ufficio, di corrispettivi non dichiarati. Pur essendo inibito l'accertamento di valore, il criterio automatico di valutazione non implicava una diversa determinazione della base imponibile, che si identificava, ai sensi del combinato disposto dell'art. 43  TUIR , comma 1, e dell'art. 51  TUR , con il "valore del bene o del diritto alla data dell'atto", assumendosi per tale "quello dichiarato dalle parti nell'atto e, in mancanza o se superiore, il corrispettivo pattuito". Per le cessioni di immobili soggette ad I.V.A., il D.L. 23 febbraio 1995, n. 41 , art. 15 , aveva esteso (per i fabbricati classificati o classificabili nei gruppi A, B e C) il principio della non rettificabilità del corrispettivo dichiarato, ove determinato in base ai parametri automatici previsti per l'imposta di registro, salvo che da atto o documento il corrispettivo risultasse di maggiore ammontare;
- il D.L. n. 223 del 2006 , art. 35 , comma 2 (cd. Decreto Visco - Bersani), convertito, con modificazioni, dalla L. n. 248 del 2006  (decreto in vigore dal 4 luglio 2006), ha inserito nel D.P.R. n. 633 del 1972 , art. 54 , comma 3 (ai fini dell'I.V.A.) una disposizione in base alla quale "per le cessioni aventi ad oggetto beni immobili e relative pertinenze, la prova di cui al precedente periodo s'intende integrata anche se l'esistenza delle operazioni imponibili o l'inesattezza delle indicazioni di cui al comma 2, sono desunte sulla base del valore normale dei predetti beni, determinato ai sensi del presente decreto, art. 14.". Lo stesso citato D.L. n. 223 del 2006 , art. 35 , comma 3, ha inoltre inserito nel D.P.R. n. 600 del 1973 , art. 39 , comma 1, lett. d), (ai fini delle imposte sui redditi) una disposizione analoga alla precedente ed in base alla quale "per le cessioni aventi ad oggetto beni immobili, ovvero la costituzione o i trasferimento di diritti reali di godimento sui medesimi beni, la prova (...) si intende integrata anche se l'infedeltà dei ricavi viene desunta sulla base del valore normale dei predetti beni determinato ai sensi dell'art. 9  TUIR , comma 3". Lo stesso citato art. 35, comma 4, ha, inoltre, espressamente abrogato il D.L. 23 febbraio 1995, n. 41 , art. 15 . Il D.L. n. 223 del 2006  ha, quindi, introdotto presunzioni semplici legali relative che consentivano all'ente impositore di rettificare la dichiarazione del contribuente sulla base del solo scostamento tra il corrispettivo dichiarato per le cessioni di beni immobili ed il valore normale degli stessi, determinato (in forza della L. n. 296 del 2006 , art. 1 , comma 307, - Legge finanziaria 2007 - e del provvedimento direttoriale del 27 luglio 2007, emesso in attuazione di tale legge e con il quale erano indicati i criteri utili per la determinazione del valore normale dei fabbricati ai sensi del D. n. 633 del 1972, art. 14, e dell'art. 9  TUIR , comma 3,) secondo i valori dell'Osservatorio del mercato immobiliare (O.M.I.) presso l'Agenzia del Territorio e i coefficienti di merito relativi alle caratteristiche dell'immobile, integrati da altre informazioni in possesso degli uffici tributari. Anche se inizialmente tali nuovi disposizioni sono state ritenute di natura "procedimentale" e, quindi, applicabili anche ad accertamenti relativi ad anni d'imposta precedenti al 4 luglio 2006 (data di entrata in vigore del decreto Visco - Bersani),la L. n. 244 del 2007 , art. 1 , comma 265, in vigore dal 10 gennaio 2008, ha stabilito che le presunzioni legali (basate sul valore normale) si applicano soltanto per gli atti formati a decorrere dal 4 luglio 2006, mentre per gli atti formati anteriormente, valgono "agli effetti tributari, come presunzioni semplici". Successivamente la Commissione Europea, nell'ambito del procedimento di infrazione n. 2007/4575, ha rilevato l'incompatibilità - in relazione all'I.V.A., ma con valutazione ritenuta estensibile dal legislatore nazionale anche alle imposte dirette - delle disposizioni introdotte dal D.L. n. 223 del 2006 , art. 35 , con la Dir. comunitaria 2006/112/CE , art. 73 , secondo cui la base imponibile I.V.A. "comprende tutto ciò che costituisce il corrispettivo versato o da versare al fornitore o al prestatore per tali operazioni da parte dell'acquirente destinatario o di un terzo, comprese le sovvenzioni direttamente connesse con il prezzo di tali operazioni". In considerazione di tale parere, la L. n. 88 del 2009 , art. 24 , comma 4, lett. f), e la L. n. 88 del 2009 , art. 5  (legge comunitaria del 2008), è nuovamente intervenuta sul citato art. 39, stabilendo all' art. 39, comma 1, lett. d): "Per i redditi d'impresa delle persone fisiche l'ufficio procede alla rettifica:(...) d) se l'incompletezza, la falsità o l'inesattezza degli elementi indicati nella dichiarazione e nei relativi allegati risulta dall'ispezione delle scritture contabili e dalle altre verifiche di cui all'art. 33, ovvero dal controllo della completezza, esattezza e veridicità delle registrazioni contabili sulla scorta delle fatture e degli altri atti e documenti relativi all'impresa nonchè dei dati e delle notizie raccolti dall'ufficio nei modi previsti dall'art. 32. L'esistenza di attività non dichiarate o l'inesistenza di passività dichiarate è desumibile anche sulla base di presunzioni semplici, purchè queste siano gravi, precise e concordanti", nonchè sul D.P.R. n. 633 del 1972 , art. 54 , comma 3, prevedendo: "L'Ufficio può tuttavia procedere alla rettifica indipendentemente dalla previa ispezione della contabilità del contribuente qualora l'esistenza di operazioni imponibili per ammontare superiore a quello indicato nella dichiarazione, o l'inesattezza delle indicazioni relative alle operazioni che danno diritto alla detrazione, risulti in modo certo e diretto, e non in via presuntiva, da verbali, questionari e fatture di cui all'art. 51, comma 2, nn. 2), 3) e 4), dagli elenchi allegati alle dichiarazioni nonchè da altri atti e documenti in suo possesso";
- questa Corte ha, quindi, ripetutamente affermato che, in tema di accertamento dei redditi d'impresa, in seguito alla sostituzione del citato art. 39 , ad opera della L. n. 88 del 2009 , art. 24, comma 5, che, con effetto retroattivo - stante la sua finalità di adeguamento al diritto dell'Unione Europea - ha eliminato la presunzione relativa di corrispondenza dei corrispettivo della cessione di beni immobili al valore normale degli stessi introdotta dal citato art. 35, così ripristinando il precedente quadro normativo in base al quale l'esistenza di attività non dichiarate può essere desunta "anche sulla base di presunzioni semplici, purchè queste siano gravi, precise e concordanti", l'accertamento di un maggior reddito derivante dalla predetta cessione di beni immobili non può essere fondato soltanto sulla sussistenza di uno scostamento tra il corrispettivo dichiarato nell'atto di compravendita ed il valore normale del bene quale risulta dalle quotazioni ma richiede la sussistenza di ulteriori elementi indiziari gravi, precisi e concordanti (Cass. n. 23379 del 2019 ; n. 9474 del 2017 ; Cass. n. 26487 del 2016 ; n. 24054 del 2014 ; Cass. n. 11439 del 2018 ; n. 2155 del 25/1/2019 );
- nella sentenza impugnata, la CTR, facendo buon governo dei suddetti principi, ha correttamente condiviso l'operato dell'Ufficio che, nel ricostruire con metodo analitico - induttivo i maggiori ricavi del contribuente in relazione alle vendite di due unità immobiliari, ha fondato l'indagine su diversi elementi indiziari - stimati precisi, gravi e concordanti - quali, oltre allo scostamento tra il corrispettivo dichiarato negli atti di vendita dalle quotazioni OMI e dalle rilevazioni delle tabelle camerali, per la compravendita di uno degli appartamenti, il valore dell'immobile indicato nella perizia dell'istituto di credito prodromica alla concessione del finanziamento e per la compravendita dell'altro uguale e contiguo appartamento la dichiarazione - non validamente contestata dal contribuente - resa dagli acquirenti dello stesso; ciò in conformità all'insegnamento di questa Corte secondo cui "La valutazione della prova presuntiva esige che il giudice di merito esamini tutti gli indizi di cui disponga non già considerandoli isolatamente, ma valutandoli complessivamente ed alla luce l'uno dell'altro, senza negare valore ad uno o più di essi sol perchè equivoci, cosi da stabilire se sia comunque possibile ritenere accettabilmente probabile l'esistenza del fatto da provare" (Cass. sez. 3, n. 5787 del 2014; v. Cass., sez. 6-5, n. 30276 dei 2017 ); in particolare, quanto alla denunciata erroneità della sentenza per avere la CTR ravvisato un elemento presuntivo grave, preciso e concordante nel preliminare di compravendita non firmato dalle parti e, pertanto, inesistente, la censura non coglie la ratio decidendi, avendo il giudice di appello, nella valutazione complessiva degli elementi indiziari, fatto riferimento, al preliminare di compravendita relativo ad uno dei due appartamenti, non sottoscritto dalle parti - reperito presso l'istituto di credito e riportante un corrispettivo superiore a quello di cui al rogito - esclusivamente quale documento sul quale era stata redatta la perizia estimativa volta a stabilire la congruità del prezzo in relazione al previsto finanziamento bancario, perizia alla quale - quale "atto tecnico, redatto da un professionista del settore" - occorreva, a suo avviso, dare rilevanza, non essendo stata validamente contestata dal contribuente; ogni altra argomentazione relativa alla assunta riferibilità del preliminare di vendita in questione a una diversa costruzione immobiliare, tende inammissibilmente a rivisitazioni di valutazioni di merito già effettuate dal giudice di appello;
- con il terzo motivo, il ricorrente denuncia, in relazione all'art. 360  c.p.c., comma 1, n. 4, la nullità della sentenza impugnata per omessa/apparente motivazione per avere la CTR - sull'eccezione relativa alla mancanza di sottoscrizione del preliminare utilizzato dall'Agenzia quale elemento presuntivo per fondare l'accertamento - apoditticamente affermato che non erano "accoglibili le riserve espresse dal contribuente in merito al fatto che il preliminare risultasse) non firmato dalle parti" senza alcuna altra argomentazione sul punto;
- il motivo è inammissibile in quanto non coglie il decisum per avere la CTR superato le riserve del contribuente in merito alla mancata sottoscrizione del preliminare di compravendita (reperito presso l'istituto di credito) relativo ad uno dei due appartamenti in considerazione del fatto che occorresse dare rilievo - unitamente agli scostamenti del prezzo indicato nel rogito dai valori Orni e dalle tabelle camerali - alla perizia estimativa - redatta in base al preliminare e volta a stabilire la congruità del prezzo in relazione al previsto finanziamento bancario - quale "atto tecnico redatto da un professionista del settore", non oggetto di valida contestazione da parte del contribuente;
- in conclusione, il ricorso va rigettato;
- le spese del giudizio di legittimità seguono la soccombenza e vengono liquidate come in dispositivo.

"""

In [ ]:
response2 = gu.ask_gpt_2(prompt_par_div(txt3, jpol_labels), system_prompt, KEY, model="gpt-4o", max_tokens=4000, temperature=0.2, top_p=1)

In [ ]:
print(response2)

# Translation

In [ ]:
file = "/content/drive/MyDrive/POLINE/xml svedesi/SE SET A/kevin/RÅ+2008+ref.+72.xml"

with open(file) as f:
    file = f.read()

txt = "".join(file)

In [ ]:
def prompt_translate(txt):
  return f"""
  Translate the following swedish text in english
  {txt}

"""

system_prompt = "You are a judge with strong knowledge on tax law"

In [ ]:
response = gu.ask_gpt_2(prompt_translate(txt), system_prompt, KEY, model="gpt-4o-mini", max_tokens=4000, temperature=0.2, top_p=1)
print(response)

**Supreme Administrative Court RÅ 2008 Ref. 72**

Case number: 2281-06  
Decision date: 2008-11-10  

**Title:**  
A limited company acquired a boat and received a full deduction for input VAT. Question of whether the company's withdrawal taxation should be based on actual use of the boat for non-business purposes or already on the right of disposal. Also, question of the placement of the burden of proof.

**Legal provisions:**  
- Chapter 2, Section 5, first paragraph 2 and Chapter 7, Section 3, paragraph 2 b of the Value Added Tax Act (1994:200)  
- Article 6.2 a and Article 11 A.1 c of the Council's Sixth Directive (77/388/EEC) of May 17, 1977, on the harmonization of member states' legislation regarding turnover taxes - Common system of value added tax: uniform basis for assessment  

**Case law:**  
- ECJ ruling in case C-97/90 Lennartz  
- ECJ ruling in case C-230/94 Enkler  
- RÅ 2001 Ref. 22  

**SUMMARY**  

Euro Business Travel & Publishing AB, which had business travel sales